<a href="https://colab.research.google.com/github/IsaacFayle-Waters/KBHS/blob/main/KBHS_Final_Essay.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#### Project Overview
Students were asked to choose a problem from a previously completed workbook from the Knowledge Based section of the module, in order to design and describe an alternate Hybrid solution. Code optional.  

In [ ]:
# @title
import os
from langchain_experimental.graph_transformers import LLMGraphTransformer
from langchain_google_genai import GoogleGenerativeAI, ChatGoogleGenerativeAI
import networkx as nx
from langchain_community.chains.graph_qa.base import GraphQAChain
from langchain_core.documents import Document
from langchain_community.graphs.networkx_graph import NetworkxEntityGraph

llm = GoogleGenerativeAI(model="gemini-2.5-flash", google_api_key=GOOGLE_API_KEY)

# Rag-Based Knowledge Graph Retrieval

Isaac Fayle-Waters

### Selected Workbook: 3.2 \- KnowledgeGraph (WB-3.2)

The exercise (Covato, 2026\) extracts entities and relationships from raw text to create a knowledge graph (KG), correlating thematically with a mini social network detailing people’s interests, professional fields, and relationships. Questions are given and answers retrieved.

This is achieved programmatically; the solution is only valid for data following a triplet pattern (`head,` `relation,` `tail`) in single sentences, separated by full stops. This pattern can be referred to as a ‘`fact`’ (Ji et al., 2022).

The program creates `nodes` and `edges` based on string positioning; splits can only occur successfully if a precise syntax of `NodeWord [Space] EdgePhrase [Space] NodeWord [Fullstop]` is present.

Limitations are even more evident when both `tail` and `head` nodes are people, creating less common ‘bi-directional’ relationships. Additional steps are then necessary: `EdgePhrase` must be specific ( `works_with` ).

### Proposed Alternative

#### RAG

Retrieval Augmented Generation (RAG) extends the behaviour and usefulness of LLMs through non-parametric information retrieval. Documents associated with a given input are retrieved and combined with the input to produce output sequences. This can avoid pitfalls like hallucinations (Lewis et al., 2020), which can stem from training \- statistical quirks or incorrect data \- or attempts to maximise reward: Some benchmarks may treat refusals and mistakes equally, so while gambling at a ‘guess’ may increase reward, refusal certainly won’t (Kalai et al., 2025).  

Within WB-3.2, a single document contains all relevant data. This might suggest that ‘retrieval’ is the most important aspect of RAG for this task. However, the task requires answers to be given to questions that require summation of the entire document: query-focused summarisation (QFS) (Edge et al., 2024).

#### GraphRag

LLMs have limited context windows, and while the document used in WB-3.2 may not exceed most modern token limits, the hybrid solution should be scalable and flexible.

GraphRag design uses LLMs to split documents into hierarchically linked ‘communities’ of contextually related nodes (‘key entities’) and edges (‘relations’). When a query is presented, partial answers are given at a community level and combined to help generate a final answer. (Edge et al., 2024).

### Solution

A simple Langchain structure was found (Gupta, 2024) and adapted to more precisely follow the program in WB-3.2.

CB 1 shows how nodes and relations are extracted from the document, using `LLMGraphTransformer` (langchain-ai, 2025), which takes a supplied LLM (gemini-2.5-flash) and uses it to convert the contents into structured output (O1 and O2).

CB 2 shows the same process with constraints relative to WB-3.2.

CB 3 shows integration with `networkx.` Nodes and edges are still defined programmatically; however, CB 1 evidences that the process of extraction can be dynamic. Either aproach solves Q1, one of which is evidenced in CB 4 and Output 3.

#### CB 1. No Structure - LLM is free to choose Nodes and Edges

In [ ]:
#Code Block 1

#Document extraction
documents = [Document(page_content=text)]
#Assign LLM to Transform Doc 2 Graph
llm_transformer = LLMGraphTransformer(llm=llm)
#Create Graph
graph_documents = llm_transformer.convert_to_graph_documents(documents)

##### CB 2. Structure Imposed. LLM Node/edge choices directed.

In [ ]:
#Code Block 2 : Structure Imposed / Filtered

#Same as Code Block 1, but with structure imposed
#Specified Filtering of Nodes and Relatioonshiups
llm_transformer_filtered = LLMGraphTransformer(
    llm=llm,
    allowed_nodes=["Person", "Activity", "Reseach_Area"],
    allowed_relationships=["likes", "researches", "works_with"],
)
graph_documents_filtered = llm_transformer_filtered.convert_to_graph_documents(
    documents
)

##### Output 1 - Node Creation
```
[GraphDocument(nodes=[Node(id='Jack', type='Person', properties={}), Node>  (id='sewing', type='Activity', properties={})]
```

##### Output 2 - Relationship Creation
fact = {Elena, works_with, Claude}
```
Relationship(source=Node(id='Elena', type='Person', properties={}), target=Node(id='Claude', type='Person', properties={}), type='works_with', properties={})
```

##### CB 3. - KG Generation

In [ ]:
#Code Block 3 - KG Generation

#Networkx integration
graph = NetworkxEntityGraph()

#Add nodes
for node in graph_documents_filtered[0].nodes:
    graph.add_node(node.id)

#Add edges
for edge in graph_documents_filtered[0].relationships:
    graph._graph.add_edge(
            edge.source.id,#Head
            edge.target.id,#Tail
            relation=edge.type,#Relation
        )

##### CB 4 and Output 3 - QFS

In [ ]:
#Code Block 4 - Q&A chain

chain = GraphQAChain.from_llm(
    llm=llm,
    graph=graph,
    verbose=True
)
question_one = """How many subjects does Elena research?"""
chain.run(question_one)



> Entering new GraphQAChain chain...
Entities Extracted:
Elena
Full Context:
Elena works_with Claude
Elena researches IoT
Elena likes basketball
Elena likes tennis
Elena works_with Ivy
Elena researches Databases

> Finished chain.


'Elena researches 2 subjects.'

### Pros

The solution allows greater flexibility regarding both the type of information that can be extracted and its form. Capabilities are generalised: Nodes and edges can be auto-defined at extraction (CB 1), so they can be applied to new documents without alteration.

The bounds of a `fact` no longer need definition: specifying a symbol is unnecessary. It also seems plausible that one-to-many (‘Zak works with Natasha and Chantelle’) relations could be extracted, though this was not tested.

### Cons

There is now a likelihood of non-deterministic output. This may be controllable by passing additional prompts into `LLMGraphTransformer` or by selecting a different base-LLM; however, gaps are possible. WB-3.2’s original code will always produce the expected output.  

More configurables can become an issue. The solution only worked for Q1, and bi-directional relationships are not automatically satisfied, meaning reconfiguration is needed. However, much of the program is hidden by wrappers and libraries; it is opaque. Similarly, issues may be LLM-specific: a stronger model might work. But this highlights the final issue: the introduction of economic costs and higher resource usage






## Ethical Considerations of the Proposal

### Legality

WB-3.2 is a toy example using only first names, but if this system were used for any real-world purpose, there is a strong possibility of PII usage. The proposal moves from a system only exposing data to the immediate development environment to one where data is used by external agents. Article 5.1.f of GDPR asserts that PII should be processed in a secure manner that adequately protects against theft using technical or organisational means (European Parliament, 2016), (UK Government, 2018).

This can be a challenge for a RAG-based system, which uses non-vectorised data. In the above, there are two points of potential weakness: document extraction and the QFS chain. There are potential remedies. PII can be removed, replaced with placeholders or synthetic data. Obfuscation can also be applied using diffraction methods, which replace and mask words (Bodea, Meisenbacher, and Matthes, 2026).  However, this can reduce usability.

RAG-Based models are susceptible to poisoning attacks due to their use of external documents. Some of these types of attack are less effective against GraphRag models due to the difference in architecture, though GraphRag-specific poisoning has been developed. Mitigation techniques have been explored: paraphrasing queries, altering prompts to exclude information without supporting evidence, and checking response consistency (liang et al., 2025).




### Ethics

The proposed model increases computation. The environmental impact of LLM instance usage surpasses the much-discussed training carbon emissions over time, so while implementing a LangChain solution may be scalable and flexible, each run of the proposed system will have an increased carbon footprint compared to the original implementation (Ilager, Briem and Brandic, 2025). Optimisation, model choice, and vector database choice, could foster greater CPU utilisation for RAG models (Şakar and Emekci, 2024), creating more efficient systems.

If the system were intended for something more sensitive, like grouping patient lifestyles (‘used to smoke’) by some `relation` (increased-chances-of) to a risk-factor (Lung Disease), the non-deterministic possibilities baked into how LLMs return output could be problematic. However, Rag-Based systems are less likely to produce hallucinations than ordinary LLMs, meaning data is less likely to be made up, endangering lives.  One study (Hussain, 2025\) focused on GraphRag specifically as a way of managing what it terms ‘values-debt’. The system made use of GraphRag for an aviation safety LLM, praising the paradigm's completeness and ability to make ‘causal links’  

## References

Bodea, A.E., Meisenbacher, S. and Matthes, F. (2026) a Case Study on the Impact of Anonymization along the RAG Pipeline. *arXiv preprint* \[online\]. Available from: https://arxiv.org/pdf/2604.15958 \[Accessed 29 April 2026\].

Covato, E. (2026) *WorkBook 3.2 \- KnowledgeGraph* . Jupyter Notebook, University of the West of England.

Edge, D. *et al.* (2024) From Local to Global: a Graph RAG Approach to Query-Focused Summarization. *arXiv preprint* \[online\]. Available from: https://arxiv.org/abs/2404.16130 \[Accessed 26 April 2026\].

European Parliament (2016) *General Data Protection Regulation* \[online\]. 5, 6\. Available from: https://gdpr-info.eu/art-5-gdpr/.

Gupta, M. (2024) *GraphRAG Using LangChain* *Medium*. 17 July 2024 \[online\]. Available from: https://medium.com/data-science-in-your-pocket/graphrag-using-langchain-31b1ef8328b9 \[Accessed 26 April 2026\].

Hussain, W. (2025) Mitigating Values Debt in Generative AI: Responsible Engineering with Graph RAG. *2025 IEEE/ACM International Workshop on Responsible AI Engineering (RAIE)* \[online\]. pp. 9–12. Available from: https://ieeexplore.ieee.org/document/11029428 \[Accessed 2 July 2025\].

Ilager, S., Briem, L.F. and Brandic, I. (2025) Green-Code: Learning to Optimize Energy Efficiency in Llm-Based Code Generation. *IEEE 25th International Symposium on Cluster, Cloud and Internet Computing (CCGrid)* \[online\]. pp. 559–569. Available from: https://ieeexplore.ieee.org/abstract/document/11044793 \[Accessed 29 April 2026\].

Ji, S., Pan, S., Cambria, E., Marttinen, P. and Yu, P.S. (2022) A Survey on Knowledge Graphs: Representation, Acquisition, and Applications. *IEEE Transactions on Neural Networks and Learning Systems* \[online\]. 33 (2), pp. 494–514. Available from: https://ieeexplore.ieee.org/abstract/document/9416312 \[Accessed 24 April 2026\].

Kalai, A.T., Nachum, O., Vempala, S. and Zhang, E. (2025) Why Language Models Hallucinate. *arXiv preprint* \[online\]. Available from: https://arxiv.org/abs/2509.04664.

LangChain (2026) *NetworkxEntityGraph | langchain\_community | LangChain Reference* *Langchain.com*. 2026 \[online\]. Available from: https://reference.langchain.com/python/langchain-community/graphs/networkx\_graph/NetworkxEntityGraph \[Accessed 29 April 2026\].

langchain-ai (2025) *langchain-experimental/libs/experimental/langchain\_experimental/graph\_transformers/llm.py GitHub*. 2025 \[online\]. Available from: https://github.com/langchain-ai/langchain-experimental/blob/main/libs/experimental/langchain\_experimental/graph\_transformers/llm.py \[Accessed 29 April 2026\].

Lewis, P. *et al.* (2020) Retrieval-Augmented Generation for Knowledge-Intensive NLP Tasks. *Advances in Neural Information Processing Systems*. 33, pp. 9459--9474.

liang, J., Wang, Y., Li, C., Zhu, R., Jiang, T., Gong, N. and Wang, T. (2025) GraphRag Under Fire. \[online\]. Available from: https://arxiv.org/pdf/2501.14050 \[Accessed 29 April 2026\].

Şakar, T. and Emekci, H. (2024) Maximizing RAG efficiency: A comparative analysis of RAG methods. *Natural Language Processing* \[online\]. pp. 1–25.

UK Government (2018) *UK General Data Protection Regulation* *legislation.gov.uk*. 2018 \[online\]. Available from: https://www.legislation.gov.uk/eur/2016/679/contents \[Accessed 29 April 2026\].

### LLM Usage
Generative AI was not used for any intellectual work or code. Some automated document summarisation was observed, though the source of this is a Google Scholar PDF Reader browser extention.   